# Tutorial 3: Crystal Generation

Generate 3D crystal structures with periodic boundary conditions.

**Time**: 30 minutes

**Topics**:
- Crystal data loading
- Molecular conditioning
- Unit cell parameter learning
- CIF file export

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import torch
import numpy as np
import matplotlib.pyplot as plt

from crystal.data import MoleculeDataset, CrystalDataset, MoleculeCrystalMapper
from crystal.models import MolecularEncoder, CrystalDynamics, CrystalDiffusion
from crystal.conditioning import MolecularConditioning, SpaceGroupEmbedding, DensityConditioning
from crystal.utils import CIFWriter, CellOperations
from crystal.evaluation import StructureValidator, CrystalMetrics

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

## 1. Create Sample Crystal Data

In [ ]:
# Create sample databases
from ase import Atoms
from ase.db import connect

# Molecule database
mol_db = connect('tutorial_molecules.db', append=False)

# Simple benzene molecule
benzene_positions = [
    [0.0, 1.4, 0.0],
    [1.21, 0.7, 0.0],
    [1.21, -0.7, 0.0],
    [0.0, -1.4, 0.0],
    [-1.21, -0.7, 0.0],
    [-1.21, 0.7, 0.0],
]
benzene = Atoms('C6', positions=benzene_positions)
mol_db.write(benzene, data={'molecule_id': 'benzene_001'})

print("Created molecule database: tutorial_molecules.db")

# Crystal database
crystal_db = connect('tutorial_crystals.db', append=False)

# Benzene crystal unit cell
cell = [7.44, 9.55, 6.92, 90.0, 90.0, 90.0]  # Orthorhombic cell

# Multiple benzene molecules in unit cell
crystal_positions = np.array(benzene_positions) / 2  # Scale down
crystal = Atoms('C6', positions=crystal_positions, cell=cell, pbc=True)

crystal_db.write(crystal, data={
    'molecule_id': 'benzene_001',
    'crystal_id': 'benzene_crystal_001',
    'space_group': 14,  # P2_1/c
    'density': 1.08,
})

print("Created crystal database: tutorial_crystals.db")
print(f"\nCell parameters: {cell}")
print(f"Space group: 14 (P2_1/c)")
print(f"Density: 1.08 g/cm³")

## 2. Load Crystal Dataset

In [ ]:
# Load datasets
mol_dataset = MoleculeDataset(
    db_path='tutorial_molecules.db',
    dataset_info={'atom_decoder': ['C', 'H', 'N', 'O', 'F']}
)

crystal_dataset = CrystalDataset(
    db_path='tutorial_crystals.db',
    molecule_dataset=mol_dataset,
    dataset_info={'atom_decoder': ['C', 'H', 'N', 'O', 'F']}
)

print(f"Loaded {len(mol_dataset)} molecules")
print(f"Loaded {len(crystal_dataset)} crystals")

# Examine crystal data
crystal_data = crystal_dataset[0]
print("\nCrystal data keys:", crystal_data.keys())
print(f"Positions shape: {crystal_data['positions'].shape}")
print(f"Cell params: {crystal_data['cell_params']}")
print(f"PBC: {crystal_data['pbc']}")
print(f"Space group: {crystal_data['space_group']}")
print(f"Density: {crystal_data['density']:.2f} g/cm³")

## 3. Setup Crystal Model

In [ ]:
# Molecular encoder
mol_encoder = MolecularEncoder(
    in_node_nf=5,  # One-hot atom types
    hidden_nf=128,
    n_layers=4,
    attention=True
).to(device)

# Molecular conditioning
mol_conditioning = MolecularConditioning(
    molecular_feature_dim=128,
    conditioning_dim=256,
    use_geometry=True  # Include size, volume, axes
).to(device)

# Optional: Space group conditioning
sg_embedding = SpaceGroupEmbedding(
    embedding_dim=64,
    num_space_groups=230
).to(device)

# Optional: Density conditioning
density_conditioning = DensityConditioning(
    conditioning_dim=64,
    min_density=0.5,
    max_density=5.0
).to(device)

print("Created conditioning modules")
print(f"Molecular encoder params: {sum(p.numel() for p in mol_encoder.parameters()):,}")
print(f"Total conditioning dim: {256 + 64 + 64}")

In [ ]:
# Crystal dynamics model
crystal_model = CrystalDynamics(
    in_node_nf=5,
    hidden_nf=128,
    context_node_nf=384,  # Sum of conditioning dims
    n_layers=6,
    attention=True,
    learn_lattice=True,
    lattice_hidden_dim=128
).to(device)

# Wrap in diffusion
crystal_diffusion = CrystalDiffusion(
    dynamics=crystal_model,
    in_node_nf=5,
    n_dims=3,
    timesteps=500,
    noise_schedule='polynomial_2',
    loss_type='l2'
).to(device)

print(f"\nCrystal model params: {sum(p.numel() for p in crystal_model.parameters()):,}")

## 4. Extract Molecular Features

In [ ]:
# Get molecule data
mol_data = mol_dataset[0]

h = mol_data['one_hot'].to(device)
x = mol_data['positions'].to(device)
edge_index = mol_data['edge_index'].to(device)
node_mask = mol_data['atom_mask'].to(device)

print(f"Molecule: {mol_data['n_nodes']} atoms")

# Extract features
with torch.no_grad():
    mol_features = mol_encoder(h, x, edge_index, node_mask)

print(f"\nExtracted molecular features:")
print(f"  Node features: {mol_features['node_features'].shape}")
print(f"  Global features: {mol_features['global_features'].shape}")
print(f"  Molecular size: {mol_features['mol_size']:.2f} Å")
print(f"  Molecular volume: {mol_features['mol_volume']:.2f} ų")
print(f"  Principal axes: {mol_features['principal_axes'].shape}")

## 5. Prepare Conditioning

In [ ]:
# Molecular conditioning (PRIMARY)
with torch.no_grad():
    mol_context = mol_conditioning(mol_features)

print(f"Molecular context: {mol_context.shape}")

# Space group conditioning
space_group = torch.tensor([14], device=device)  # P2_1/c
with torch.no_grad():
    sg_context = sg_embedding(space_group)

print(f"Space group context: {sg_context.shape}")

# Density conditioning
density = torch.tensor([1.08], device=device)
with torch.no_grad():
    density_context = density_conditioning(density)

print(f"Density context: {density_context.shape}")

# Combine all conditioning
combined_context = torch.cat([mol_context, sg_context, density_context], dim=-1)
print(f"\nCombined context: {combined_context.shape}")
print("Ready for crystal generation!")

## 6. Generate Crystal (Demo)

In [ ]:
print("Crystal generation demo")
print("\nNote: For actual generation, model needs to be trained.")
print("This shows the interface and data flow.\n")

# Setup for sampling
n_samples = 1
n_atoms = 12  # Atoms in crystal unit cell

node_mask = torch.ones(n_samples, n_atoms, 1, device=device)
edge_mask = torch.ones(n_samples, n_atoms, n_atoms, device=device)

# Initial cell parameters (will be refined)
cell_params = torch.tensor([[7.0, 9.0, 7.0, 90.0, 90.0, 90.0]], device=device)
pbc = torch.tensor([[True, True, True]], device=device)

# Expand context for all atoms
context = combined_context.unsqueeze(0).expand(n_samples, n_atoms, -1)

print(f"Sampling setup:")
print(f"  Samples: {n_samples}")
print(f"  Atoms: {n_atoms}")
print(f"  Context shape: {context.shape}")
print(f"  Initial cell: {cell_params[0].tolist()}")
print(f"\nFor trained model:")
print("  x, h, cell = crystal_diffusion.sample(...)")

## 7. Cell Operations

In [ ]:
# Cell parameter utilities
cell_params_np = cell_params[0].cpu().numpy()
a, b, c, alpha, beta, gamma = cell_params_np

# Convert to matrix
cell_matrix = CellOperations.params_to_matrix(a, b, c, alpha, beta, gamma)
print("Cell matrix:")
print(cell_matrix)

# Compute volume
volume = CellOperations.compute_volume(cell_params=cell_params_np)
print(f"\nCell volume: {volume:.2f} ų")

# Convert back
params_back = CellOperations.matrix_to_params(cell_matrix)
print(f"\nRound-trip check:")
print(f"  Original: {cell_params_np}")
print(f"  Recovered: {np.array(params_back)}")
print(f"  Max error: {np.abs(cell_params_np - params_back).max():.6f}")

## 8. Structure Validation

In [ ]:
# Validate crystal structure
validator = StructureValidator()

# Example structure
structure = {
    'positions': crystal_data['positions'].cpu().numpy(),
    'atom_types': crystal_data['one_hot'].argmax(dim=-1).cpu().numpy(),
    'cell_params': crystal_data['cell_params'].cpu().numpy(),
    'pbc': crystal_data['pbc'].cpu().numpy(),
}

is_valid, errors = validator.validate_structure(structure)

print(f"Structure validation:")
print(f"  Valid: {is_valid}")
if errors:
    print(f"  Errors: {errors}")
else:
    print(f"  ✓ All checks passed")

# Check specific constraints
print(f"\nValidation checks:")
print(f"  ✓ Cell parameters: {structure['cell_params']}")
print(f"  ✓ Positive lengths: {all(structure['cell_params'][:3] > 0)}")
print(f"  ✓ Valid angles: {all((0 < structure['cell_params'][3:]) & (structure['cell_params'][3:] < 180))}")
print(f"  ✓ Positions shape: {structure['positions'].shape}")

## 9. CIF Export

In [ ]:
# Export to CIF format
dataset_info = {'atom_decoder': ['C', 'H', 'N', 'O', 'F']}
writer = CIFWriter(dataset_info)

# Write CIF
output_file = 'tutorial_crystal.cif'
writer.write_cif(
    structure,
    output_file,
    compound_name='Tutorial Benzene Crystal',
    space_group=14
)

print(f"CIF file written: {output_file}")

# Display CIF content
print("\nCIF file content:")
with open(output_file, 'r') as f:
    print(f.read())

## 10. Crystal Metrics

In [ ]:
# Compute crystal metrics
metrics_obj = CrystalMetrics(dataset_info)

# Single crystal metrics
metrics = metrics_obj.compute_structure_metrics(structure)

print("Crystal metrics:")
print(f"  Volume: {metrics['volume']:.2f} ų")
print(f"  Density: {metrics['density']:.2f} g/cm³")
print(f"  Cell params: {metrics['cell_params']}")

# Batch metrics
structures = [structure]  # List of structures
batch_metrics = metrics_obj.compute_all_metrics(structures)

print("\nBatch metrics:")
print(f"  Validity ratio: {batch_metrics['validity_ratio']:.2%}")
print(f"  Mean volume: {batch_metrics['volume_mean']:.2f} ų")
print(f"  Mean density: {batch_metrics['density_mean']:.2f} g/cm³")

## Summary

### Learned:
1. ✅ Crystal data loading with ASE
2. ✅ Molecular feature extraction
3. ✅ Multi-modal conditioning (molecule + space group + density)
4. ✅ Cell parameter operations
5. ✅ Structure validation
6. ✅ CIF file export
7. ✅ Crystal metrics

### Training Command:
```bash
python main_crystal.py \
    --exp_name my_crystal \
    --crystal_db_path tutorial_crystals.db \
    --molecule_db_path tutorial_molecules.db \
    --condition_on_molecule True \
    --condition_on_space_group True \
    --condition_on_density True \
    --batch_size 16 \
    --n_epochs 200 \
    --learn_lattice True
```

### Key Features:
- **E(3) Equivariance**: Maintained with periodic boundaries
- **No Fallbacks**: All validations explicit
- **Cell Learning**: Unit cell parameters learned during diffusion
- **Multi-Conditioning**: Molecule (required) + space group + density